In [1]:
import pandas as pd

HP_IN  = "hp_LINCS.csv"          # input
HP_OUT = "hp_LINCS_patched.csv"  # output

BASE_TOPN = "30"                 # copy from this top_n
NEW_TOPNS = ["5", "10", "15", "20", "25"]
MODS = ["GE", "Early Fusion"]    # patch only these (CP는 그대로)

df = pd.read_csv(HP_IN)

# sanity
need_cols = {"top_n","modality","hidden","act","dropout","wd"}
missing = need_cols - set(df.columns)
if missing:
    raise ValueError(f"hp csv missing columns: {sorted(missing)}")

df["top_n"] = df["top_n"].astype(str)
df["modality"] = df["modality"].astype(str).replace({"EF": "Early Fusion"})

base = df[(df["top_n"] == BASE_TOPN) & (df["modality"].isin(MODS))].copy()
if base.empty:
    raise ValueError(f"No rows found for top_n={BASE_TOPN} and modality in {MODS}")

new_rows = []
for tn in NEW_TOPNS:
    for mod in MODS:
        exists = ((df["top_n"] == tn) & (df["modality"] == mod)).any()
        if exists:
            continue
        r = base[base["modality"] == mod].copy()
        if r.empty:
            raise ValueError(f"Missing base row for modality={mod} at top_n={BASE_TOPN}")
        r.loc[:, "top_n"] = tn
        new_rows.append(r)

if new_rows:
    df2 = pd.concat([df] + new_rows, ignore_index=True)
else:
    df2 = df.copy()

# keep stable ordering: numeric top_n first, then modality
def _topn_key(x):
    try:
        return int(x)
    except Exception:
        return 10**9

df2["_topn_int"] = df2["top_n"].map(_topn_key)
df2 = df2.sort_values(["_topn_int","modality"]).drop(columns=["_topn_int"]).reset_index(drop=True)

df2.to_csv(HP_OUT, index=False)
print(f"[saved] {HP_OUT}")
print("added rows:", len(df2) - len(df))

[saved] hp_LINCS_patched.csv
added rows: 0
